# OSM: establecimientos de salud

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/osm_salud.geojson`. Una fila representa un objeto OSM etiquetado como hospital o clínica; no una cama ni un servicio operativo verificado.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/osm_salud.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '04_osm_salud', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Tipo de establecimiento y cobertura de atributos operativos

In [ ]:
for campo in ["amenity","emergency","operational_status","healthcare","source","earthquake:damage","damage:type"]:
 if campo in df:display(conteos(df[campo],campo).head(15))
capacidad=faltantes(df,["name","emergency","operational_status","beds","capacity","healthcare:speciality","check_date","opening_hours"])
display(capacidad)
grafico_barras(conteos(df.amenity,"tipo"),"Tipos registrados en OSM", "04_tipos")
resumen.update({"tipos": df.amenity.value_counts().to_dict(),
                "urgencias":df.emergency.fillna("Sin dato").value_counts().to_dict(),
                "operatividad":df.operational_status.fillna("Sin dato").value_counts().to_dict(),
                "capacidad":capacidad.to_dict("records"),
                "con_etiqueta_dano_sismico":int(df.get("earthquake:damage",pd.Series(dtype=str)).notna().sum())})

## 4. Duplicados candidatos y coordenadas coincidentes

In [ ]:
from scipy.spatial import cKDTree
norm=df["name"].fillna("").map(lambda s: re.sub(r"\W+","",unicodedata.normalize("NFKD",s).encode("ascii","ignore").decode().lower()))
xy=np.column_stack([g.to_crs(CRS_METROS).geometry.x,g.to_crs(CRS_METROS).geometry.y])
pares=cKDTree(xy).query_pairs(30)
pares_mismo_nombre=[(a,b) for a,b in pares if norm.iloc[a] and norm.iloc[a]==norm.iloc[b]]
calidad_llaves={"osm_id_duplicados":int(df.osm_id.duplicated().sum()),
                "geometrias_coincidentes":geo["duplicadas_exactas"],
                "pares_nombre_normalizado_igual_hasta_30m":len(pares_mismo_nombre)}
display(pd.Series(calidad_llaves).to_frame("valor"))
print("30 m es una regla de revisión propuesta, no una prueba de que dos objetos son el mismo establecimiento.")
resumen["duplicados_candidatos"]=calidad_llaves

## Reglas decididas para plata

- **Incorporar para proximidad a sitios de salud**, con estado de capacidad desconocida cuando falten atributos.
- Crear `tipo_establecimiento` y `urgencias_declaradas` con tres estados (sí/no/sin dato). Un faltante no es un «no».
- Calcular por separado distancia a cualquier sitio y a sitios que declaran urgencias. No llamar hospital operativo a los 439 objetos ni inferir capacidad de atención.
- Mantener etiquetas de daño y su fecha: el corte posterior al evento puede incorporar sus efectos, por lo que no representa automáticamente infraestructura previa.
- Revisar pares candidatos de duplicación; no deduplicar por coordenada o proximidad únicamente. Conservar los `osm_id` originales.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('04_osm_salud', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")